## README:

* Cells labeled with '## ----- CONFIG ----- ##' contain parameters that need to be set manually

In [23]:
%%capture
!pip install FinRL
!pip install stable_baselines3
!pip install alpaca_trade_api
!pip install exchange_calendars
!pip install stockstats
!pip install wrds
!pip install websockets
!pip install yfinance
!pip install ta
!pip install sb3_contrib

In [24]:
%%capture
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip uninstall -y mamba-ssm causal-conv1d
!pip install causal-conv1d==1.4.0
!pip install mamba-ssm==2.2.2

In [25]:
%%capture
!pip install dotenv

In [26]:
import os
import sys
# add project root to path
# Assuming module folders are directly in the current working directory (e.g., /content)
root_path = os.getcwd()
sys.path.append(root_path)

In [27]:
import os
import sys
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from finrl.meta.preprocessor.preprocessors import data_split
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold

from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3 import PPO
from gymnasium import spaces

from trading_environment.custom_env import StockTradingEnv
from evaluation.backtest import run_backtest
from evaluation.visualize import plot_trades_on_price_from_backtest

## Load env variables

In [28]:
load_dotenv(r'../.env')

DATA_DIR = os.getenv('DATA_DIR')

## Train-test split and normalization

In [29]:
## ----- CONFIG ----- ##
SPLIT_DATE = '2018-01-01' # format %Y-%m-%d
DATA_FILE_NAME = 'data_with_features.csv'

In [30]:
# load finalized data
if DATA_DIR is None:
    DATA_DIR = '/content'
data_df = pd.read_csv(os.path.join(DATA_DIR, DATA_FILE_NAME))
data_df['date'] = pd.to_datetime(data_df['date'], format='%Y-%m-%d')
data_df.shape

(60543, 26)

In [31]:
SPLIT_DATE = pd.to_datetime(SPLIT_DATE, format='%Y-%m-%d')

# train-test split
min_date = data_df['date'].min()
max_date = data_df['date'].max()

train_df = data_split(data_df, start=min_date, end=SPLIT_DATE)
trade_df = data_split(data_df, start=SPLIT_DATE + pd.Timedelta(days=1), end=max_date)

print(train_df.shape, trade_df.shape)

(42606, 26) (17928, 26)


In [32]:
from sklearn.preprocessing import StandardScaler

## ----- CONFIG ----- ##
stock_features = [
    'macd',
    'macd_hist',
    'cci_30',
]

economy_features = [
    'CPIAUCSL',
    'DFF',
    'ICSA',
    'T10Y2Y',
    'VIXCLS',
    'friday',
    'month_sin',
    'month_cos'
]
# Get features to normalize
features_to_normalize = stock_features + economy_features

# Initialize StandardScaler
scaler = StandardScaler()

# Fit scaler only on training data features to avoid data leakage
train_df[features_to_normalize] = scaler.fit_transform(train_df[features_to_normalize])

# Transform both training and trading data features using the fitted scaler
trade_df[features_to_normalize] = scaler.transform(trade_df[features_to_normalize])


# check for NAs
print('Any missing values in training:', train_df.isna().any().any())
print('Any missing values in testing:', trade_df.isna().any().any())

Any missing values in training: False
Any missing values in testing: False


## Environment configuration

In [33]:
## ----- CONFIG ----- ##
price = 'close'         # name of price to use for transactions / rewards
hmax = 100              # maximum number of stocks to transact at each step
initial_amount = 5000   # initial amount of cash
buy_cost_pct = 0.005    # % cost of each stock purchase
sell_cost_pct = 0.005   # % cost of each stock sale
num_stock_shares = 0    # number of starting shares
window_size = 100       # number of previous time steps to encode
reward_scaling = 1      # reward scaling

In [34]:
# environment kwargs
stock_dim = len(data_df['tic'].unique())
state_space = 1 + 2 * stock_dim + len(stock_features) * stock_dim + len(economy_features)

env_kwargs = {
    "price": price,
    "hmax": hmax,
    "initial_amount": initial_amount,
    "buy_cost_pct": [buy_cost_pct] * stock_dim,
    "sell_cost_pct": [sell_cost_pct] * stock_dim,
    "stock_dim": stock_dim,
    "state_space": state_space,
    "action_space": stock_dim,
    "reward_scaling": 1,
    "num_stock_shares": [num_stock_shares] * stock_dim,
    "price": "close",
    "stock_features": stock_features,
    "economy_features": economy_features,
    "window_size": window_size
}


#Mamba Modeling

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from gymnasium import spaces
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from mamba_ssm import Mamba

class MambaEncoder(nn.Module):
    def __init__(self, input_dim, d_model, n_layers, dropout):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.dropout = nn.Dropout(dropout)
        # Using d_conv=4 inside Mamba, but we'll pad to at least 16 to be safe
        self.mamba_blocks = nn.ModuleList([
            Mamba(d_model=d_model, d_state=64, d_conv=4, expand=4)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.min_seq_len = 16

    def forward(self, x):
        # 1. Cast to float32 to avoid dtype mismatches causing NaNs
        x = x.float()

        # 2. Handle 2D input (batch, dim) -> (batch, 1, dim)
        if x.dim() == 2:
            x = x.unsqueeze(1)

        # 3. Check sequence length and pad if necessary
        # x is (batch, seq_len, input_dim)
        seq_len = x.shape[1]
        if seq_len < self.min_seq_len:
            pad_amount = self.min_seq_len - seq_len
            # F.pad format for (N, L, C): (pad_last_dim_left, pad_last_dim_right, pad_2nd_last_left, ...)
            # We want to pad the sequence dimension (dim 1) on the left (past)
            # Dimensions are: (batch, seq, feature)
            # Padding tuple: (pad_feature_left, pad_feature_right, pad_seq_left, pad_seq_right)
            x = F.pad(x, (0, 0, pad_amount, 0), mode='constant', value=0)

        # Forward pass
        x = self.input_proj(x)
        x = self.dropout(x)
        for block in self.mamba_blocks:
            x = block(x)
        x = self.norm(x)

        # Return the feature of the last time step
        return x[:, -1, :]

### Mamba Feature Extractor

In [36]:
class MambaFeatureExtractor(BaseFeaturesExtractor):
    def __init__(
        self,
        observation_space: spaces.Box,
        window_size: int,
        input_dim: int,
        d_model: int = 128,
        n_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__(observation_space, features_dim=d_model)
        self.mamba_encoder = MambaEncoder(
            input_dim=input_dim,
            d_model=d_model,
            n_layers=n_layers,
            dropout=dropout,
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.mamba_encoder(observations)

## Model training

In [37]:
## ----- CONFIG ----- ##
# Mamba Hyperparameters
d_model = 128  # Dimension of the Mamba model (output dimension of MambaEncoder)
n_layers = 2   # Number of Mamba blocks in the encoder
dropout = 0.1  # Dropout rate for regularization

# Redefine policy kwargs
mamba_policy_kwargs = dict(
    features_extractor_class=MambaFeatureExtractor,
    features_extractor_kwargs=dict(
        window_size=window_size,
        input_dim=input_dim,
        d_model=d_model,
        n_layers=n_layers,
        dropout=dropout
    ),
    net_arch=[64, 64]
)

In [38]:
import importlib
import trading_environment.custom_env
from stable_baselines3.common.vec_env import VecNormalize

# Reload the module to apply changes
importlib.reload(trading_environment.custom_env)

# Re-build training env with the updated StockTradingEnv and VecNormalize
env_train_vec = DummyVecEnv([lambda: Monitor(trading_environment.custom_env.StockTradingEnv(df=train_df, **env_kwargs))])
env_train_vec = VecNormalize(env_train_vec, norm_obs=True, norm_reward=True, clip_obs=10.0)

# Re-build trading/backtest env with VecNormalize (syncing stats would be ideal for eval, but initializing separate for now to ensure it runs)
env_trade_vec = DummyVecEnv([lambda: Monitor(trading_environment.custom_env.StockTradingEnv(df=trade_df, **env_kwargs))])
# For evaluation, we typically don't update normalization stats and don't normalize rewards,
# but here we just ensure it doesn't crash.
# Using separate normalization for trade env for simplicity in this step.
env_trade_vec = VecNormalize(env_trade_vec, norm_obs=True, norm_reward=False, clip_obs=10.0, training=True)

print("Environments re-initialized with VecNormalize.")

# Initialize the PPO model with MlpPolicy and Mamba arguments
model_mamba = PPO(
    policy="MlpPolicy",
    env=env_train_vec,
    policy_kwargs=mamba_policy_kwargs,
    verbose=1,
    tensorboard_log="./mamba_ppo_tb/",
    learning_rate=3e-4
)

# Define callbacks
stop_callback = StopTrainingOnRewardThreshold(reward_threshold=5000)
eval_callback = EvalCallback(
    env_train_vec,
    callback_on_new_best=stop_callback,
    eval_freq=1000,
    best_model_save_path='./logs/',
    log_path='./logs/',
    verbose=1
)

# Train the model
print("Starting training...")
model_mamba.learn(total_timesteps=50_000, callback=eval_callback)
print("Training finished.")

Environments re-initialized with VecNormalize.
Using cuda device
Starting training...
Logging to ./mamba_ppo_tb/PPO_3


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=18541.28 +/- 1008.32
Episode length: 4634.00 +/- 0.00
---------------------------------
| eval/              |          |
|    mean_ep_length  | 4.63e+03 |
|    mean_reward     | 1.85e+04 |
| time/              |          |
|    total_timesteps | 1000     |
---------------------------------
New best mean reward!
Training finished.


## Quick evaluation

In [39]:
trade_env = StockTradingEnv(df=trade_df, **env_kwargs)

In [41]:
res = run_backtest(
    model_mamba,
    trade_env,
    env_kwargs
)

ValueError: Expected parameter loc (Tensor of shape (1, 9)) of distribution Normal(loc: torch.Size([1, 9]), scale: torch.Size([1, 9])) to satisfy the constraint Real(), but found invalid values:
tensor([[nan, nan, nan, nan, nan, nan, nan, nan, nan]], device='cuda:0')

In [ ]:
%matplotlib inline
tickers = trade_df['tic'].unique()

for tic in tickers:
    fig, ax = plot_trades_on_price_from_backtest(res, ticker=tic)